In [1]:
import torch
from torch import nn
from torch.optim import SGD
import math
import sys
import os
import csv
import numpy as np
import sqlite3
import optuna
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

C:\Users\chail\anaconda3\envs\task2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_file = 'train_20M.csv'
test_file = 'test_20M.csv'
database_name = 'task2.db'

In [ ]:
def create_and_load_data(database_name, train_file, test_file):
    conn = sqlite3.connect(database_name)
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS train_data (user_id INTEGER, item_id INTEGER, rating REAL, timestamp INTEGER)''')
    cursor.execute('''CREATE TABLE IF NOT EXISTS test_data (user_id INTEGER, item_id INTEGER, timestamp INTEGER)''')
        
    with open(train_file, 'r') as train_csv:
        reader = csv.reader(train_csv)
        for row in reader:
            cursor.execute("INSERT INTO train_data VALUES (?,?,?,?)", row)
        with open(test_file, 'r') as test_csv:
            reader = csv.reader(test_csv)
            for row in reader:
                cursor.execute("INSERT INTO test_data (user_id, item_id, timestamp) VALUES (?,?,?)", (row[0], row[1], row[2]))
        conn.commit()
        conn.close()


In [ ]:
create_and_load_data(database_name, train_file, test_file)

In [3]:
conn = sqlite3.connect(database_name)
cursor = conn.cursor()

In [4]:
cursor.execute("SELECT COUNT(DISTINCT user_id) FROM train_data")
num_users = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(DISTINCT item_id) FROM train_data")
num_items = cursor.fetchone()[0]
cursor.execute("SELECT DISTINCT rating FROM train_data Order By rating")
rating_values = [row[0] for row in cursor.fetchall()]
print("Users:", num_users)
print("Items:", num_items)
print("Rating Value:",rating_values)

Users: 138493
Items: 25876
Rating Value: [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]


In [5]:
cursor.execute("SELECT COUNT(DISTINCT user_id) FROM test_data WHERE user_id NOT IN (SELECT DISTINCT user_id FROM train_data)")
cold_users = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(DISTINCT item_id) FROM test_data WHERE item_id NOT IN (SELECT DISTINCT item_id FROM train_data)")
cold_items = cursor.fetchone()[0]
print("Cold-start users:", cold_users)
print("Cold-start items:", cold_items)

Cold-start users: 0
Cold-start items: 868


In [5]:
cursor.execute("SELECT DISTINCT user_id FROM train_data")
unique_user_idx = [row[0] for row in cursor.fetchall()]

cursor.execute("SELECT DISTINCT item_id FROM train_data")
unique_item_idx = [row[0] for row in cursor.fetchall()]

num_users = len(unique_user_idx)
num_items = len(unique_item_idx)


In [28]:
train_user_tensor = torch.tensor(
    [user_to_idx[row[0]] for row in train_split],
    dtype=torch.long
)

train_item_tensor = torch.tensor(
    [item_to_idx[row[1]] for row in train_split],
    dtype=torch.long
)

train_rating_tensor = torch.tensor(
    [row[2] for row in train_split],
    dtype=torch.float32
)

val_user_tensor = torch.tensor(
    [user_to_idx[row[0]] for row in val_split],
    dtype=torch.long
)

val_item_tensor = torch.tensor(
    [item_to_idx[row[1]] for row in val_split],
    dtype=torch.long
)

val_rating_tensor = torch.tensor(
    [row[2] for row in val_split],
    dtype=torch.float32
)

global_avg = sum(row[2] for row in train_split) / len(train_split)

In [7]:
def load_train_data(database_name):
    conn = sqlite3.connect(database_name)
    cursor = conn.cursor()
    cursor.execute("SELECT user_id, item_id, rating FROM train_data")
    train_data = cursor.fetchall()
    conn.close()
    return train_data

def load_test_data(database_name):
    conn = sqlite3.connect(database_name)
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM test_data")
    test_data = cursor.fetchall()
    conn.close()
    return test_data

def verify_data(train_data, test_data):
    print("Training data sample:")
    print(train_data[:5])
    print("\nTesting data sample:")
    print(test_data[:5])

In [8]:
train_data = load_train_data(database_name)
test_data = load_test_data(database_name)
verify_data(train_data, test_data)
train_split, val_split = train_test_split(
    train_data,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

Training data sample:
[(114534, 555, 4.5), (5427, 25, 3.0), (106695, 2174, 3.0), (57384, 1020, 3.0), (82578, 259, 2.5)]

Testing data sample:
[(40131, 3499, 1238964717), (61667, 66097, 1400194424), (26940, 3168, 1112135533), (136620, 2657, 1017946298), (101273, 4226, 1293995537)]


In [16]:
class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, num_factors):
        super(MatrixFactorization, self).__init__()
        self.user_factors = nn.Embedding(num_users, num_factors)
        self.item_factors = nn.Embedding(num_items, num_factors)
        self.user_factors.weight.data.uniform_(0.25, 0.5)
        self.item_factors.weight.data.uniform_(0.25, 0.5)

    def forward(self, user_ids, item_ids):
        user_embedding = self.user_factors(user_ids)
        item_embedding = self.item_factors(item_ids)

        prediction = (user_embedding * item_embedding).sum(dim=1)  
        return prediction

In [49]:
def train_model(
    model, 
    num_epochs, 
    train_user_tensor, 
    train_item_tensor, 
    train_rating_tensor,
    lr=0.01, 
    device=None
):

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)

    train_user_tensor = train_user_tensor.to(device)
    train_item_tensor = train_item_tensor.to(device)
    train_rating_tensor = train_rating_tensor.to(device)

    loss_function = nn.MSELoss()
    optimizer = SGD(model.parameters(), lr=lr)

    num_samples = len(train_rating_tensor)
    batch_size = num_samples // 10

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        num_batches = 0

        for start in range(0, num_samples, batch_size):
            end = min(start + batch_size, num_samples)

            batch_user = train_user_tensor[start:end]
            batch_item = train_item_tensor[start:end]
            batch_rating = train_rating_tensor[start:end]

            optimizer.zero_grad()

            predictions = model(batch_user, batch_item)
            loss = loss_function(predictions, batch_rating)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

        avg_loss = total_loss / num_batches

        if epoch % 100 == 0 or epoch == num_epochs - 1:
            print(
                f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}",
                flush=True
            )

    return model


def objective(trial):
    num_factors = trial.suggest_int("num_factors", 1, 30)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = MatrixFactorization(
        num_users,
        num_items,
        num_factors,
    )

    model = train_model(
        model=model,
        num_epochs=100,
        train_user_tensor=train_user_tensor,
        train_item_tensor=train_item_tensor,
        train_rating_tensor=train_rating_tensor,
        lr=0.01,
        device=device
    )

    model.eval()

    loss_function = nn.MSELoss()

    with torch.no_grad():
        val_user_tensor_device = val_user_tensor.to(device)
        val_item_tensor_device = val_item_tensor.to(device)
        val_rating_tensor_device = val_rating_tensor.to(device)

        val_predictions = model(
            val_user_tensor_device,
            val_item_tensor_device
        )

        val_loss = loss_function(
            val_predictions,
            val_rating_tensor_device
        )

    return val_loss.item()

In [50]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print(study.best_params)
print(study.best_value)

[I 2026-05-10 20:02:20,145] A new study created in memory with name: no-name-7a52ac8f-b4d7-4603-a3be-d73e1add139d


Epoch 1/100, Loss: 11.8312
Epoch 100/100, Loss: 10.4946


[I 2026-05-10 20:02:25,869] Trial 0 finished with value: 11.543320655822754 and parameters: {'num_factors': 2}. Best is trial 0 with value: 11.543320655822754.


Epoch 1/100, Loss: 1.3218
Epoch 100/100, Loss: 1.2897


[I 2026-05-10 20:02:41,435] Trial 1 finished with value: 1.4182325601577759 and parameters: {'num_factors': 29}. Best is trial 1 with value: 1.4182325601577759.


...

Epoch 1/100, Loss: 4.0077
Epoch 100/100, Loss: 3.4702


[I 2026-05-10 20:06:19,037] Trial 18 finished with value: 3.8158960342407227 and parameters: {'num_factors': 13}. Best is trial 9 with value: 1.1238089799880981.


Epoch 1/100, Loss: 1.6107
Epoch 100/100, Loss: 1.4260


[I 2026-05-10 20:06:32,160] Trial 19 finished with value: 1.568179965019226 and parameters: {'num_factors': 20}. Best is trial 9 with value: 1.1238089799880981.


{'num_factors': 25}
1.1238089799880981


In [51]:
def train_model_until_lowest_loss(
    model,
    train_user_tensor,
    train_item_tensor,
    train_rating_tensor,
    val_user_tensor,
    val_item_tensor,
    val_rating_tensor,
    lr=0.01,
    patience=50,
    min_delta=0.0001,
    device=None
):

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)

    train_user_tensor = train_user_tensor.to(device)
    train_item_tensor = train_item_tensor.to(device)
    train_rating_tensor = train_rating_tensor.to(device)

    val_user_tensor = val_user_tensor.to(device)
    val_item_tensor = val_item_tensor.to(device)
    val_rating_tensor = val_rating_tensor.to(device)

    loss_function = nn.MSELoss()
    optimizer = SGD(model.parameters(), lr=lr)

    num_samples = len(train_rating_tensor)
    batch_size = max(num_samples // 10, 1)

    best_val_loss = float("inf")
    best_epoch = 0
    patience_counter = 0
    epoch = 0

    while True:
        epoch += 1

        model.train()

        total_loss = 0.0
        num_batches = 0

        for start in range(0, num_samples, batch_size):
            end = min(start + batch_size, num_samples)

            batch_user = train_user_tensor[start:end]
            batch_item = train_item_tensor[start:end]
            batch_rating = train_rating_tensor[start:end]

            optimizer.zero_grad()

            predictions = model(batch_user, batch_item)

            loss = loss_function(predictions, batch_rating)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

        avg_train_loss = total_loss / num_batches

        # validation
        model.eval()

        with torch.no_grad():
            val_predictions = model(val_user_tensor, val_item_tensor)

            val_loss = loss_function(
                val_predictions,
                val_rating_tensor
            ).item()

        if epoch % 100 == 0:
            print(
                f"Epoch {epoch}, "
                f"Train Loss: {avg_train_loss:.6f}, "
                f"Val Loss: {val_loss:.6f}"
            )
        if best_val_loss - val_loss > min_delta:
            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(
                f"Stopped at epoch {epoch}. "
                f"Best epoch: {best_epoch}, "
                f"Best Val Loss: {best_val_loss:.6f}"
            )
            break

    return model, best_val_loss, best_epoch

In [53]:
model = MatrixFactorization(
    num_users,
    num_items,
    25,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model, best_val_mae, best_epoch = train_model_until_lowest_loss(
    model,
    train_user_tensor,
    train_item_tensor,
    train_rating_tensor,
    val_user_tensor,
    val_item_tensor,
    val_rating_tensor,
    lr=0.01,
    patience=50,
    min_delta=0.0001,
    device=device
)

Epoch 100, Train Loss: 1.020980, Val Loss: 1.122596
Epoch 200, Train Loss: 1.005781, Val Loss: 1.105901
Epoch 300, Train Loss: 0.993153, Val Loss: 1.092029
Epoch 400, Train Loss: 0.982477, Val Loss: 1.080302
Epoch 500, Train Loss: 0.973306, Val Loss: 1.070228
Epoch 600, Train Loss: 0.965315, Val Loss: 1.061448
Epoch 700, Train Loss: 0.958262, Val Loss: 1.053699
Epoch 800, Train Loss: 0.951965, Val Loss: 1.046780
Epoch 900, Train Loss: 0.946288, Val Loss: 1.040542
Epoch 1000, Train Loss: 0.941124, Val Loss: 1.034867
Epoch 1100, Train Loss: 0.936392, Val Loss: 1.029666
Epoch 1200, Train Loss: 0.932026, Val Loss: 1.024867
Epoch 1300, Train Loss: 0.927974, Val Loss: 1.020414
Epoch 1400, Train Loss: 0.924195, Val Loss: 1.016259
Epoch 1500, Train Loss: 0.920653, Val Loss: 1.012365
Epoch 1600, Train Loss: 0.917322, Val Loss: 1.008702
Epoch 1700, Train Loss: 0.914176, Val Loss: 1.005243
Epoch 1800, Train Loss: 0.911196, Val Loss: 1.001966
Epoch 1900, Train Loss: 0.908366, Val Loss: 0.998854
Ep

In [11]:
cursor.execute("SELECT DISTINCT user_id FROM train_data")
unique_user_idx = [row[0] for row in cursor.fetchall()]

cursor.execute("SELECT DISTINCT item_id FROM train_data")
unique_item_idx = [row[0] for row in cursor.fetchall()]

num_users = len(unique_user_idx)
num_items = len(unique_item_idx)

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(unique_user_idx)
}
item_to_idx = {
    item_id: idx
    for idx, item_id in enumerate(unique_item_idx)
}

train_user_tensor = torch.tensor(
    [user_to_idx[row[0]] for row in train_data],
    dtype=torch.long
)

train_item_tensor = torch.tensor(
    [item_to_idx[row[1]] for row in train_data],
    dtype=torch.long
)

train_rating_tensor = torch.tensor(
    [row[2] for row in train_data],
    dtype=torch.float32
)
cursor.execute(
    "SELECT user_id, AVG(rating) FROM train_data GROUP BY user_id"
)

user_avg_dict = {
    row[0]: row[1]
    for row in cursor.fetchall()
}

cursor.execute(
    "SELECT item_id, AVG(rating) FROM train_data GROUP BY item_id"
)

item_avg_dict = {
    row[0]: row[1]
    for row in cursor.fetchall()
}

cursor.execute("SELECT AVG(rating) FROM train_data")

global_avg = cursor.fetchone()[0]

### Cross Validation

In [12]:
def train_model_fixed_epochs(
    model,
    train_user_tensor,
    train_item_tensor,
    train_rating_tensor,
    num_epochs=28056,
    lr=0.01,
    device=None
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)

    train_user_tensor = train_user_tensor.to(device)
    train_item_tensor = train_item_tensor.to(device)
    train_rating_tensor = train_rating_tensor.to(device)

    loss_function = nn.MSELoss()
    optimizer = SGD(model.parameters(), lr=lr)

    num_samples = len(train_rating_tensor)
    batch_size = max(num_samples // 10, 1)

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0.0
        num_batches = 0

        for start in range(0, num_samples, batch_size):
            end = min(start + batch_size, num_samples)

            batch_user = train_user_tensor[start:end]
            batch_item = train_item_tensor[start:end]
            batch_rating = train_rating_tensor[start:end]

            optimizer.zero_grad()

            predictions = model(batch_user, batch_item)
            loss = loss_function(predictions, batch_rating)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

        if epoch % 1000 == 0:
            avg_loss = total_loss / num_batches
            print(f"Epoch {epoch}/{num_epochs}, Train Loss: {avg_loss:.6f}")

    return model

def cross_validate_mf(
    train_data,
    num_users,
    num_items,
    user_to_idx,
    item_to_idx,
    user_avg_dict,
    item_avg_dict,
    global_avg,
    num_factors=25,
    num_epochs=28056,
    lr=0.01,
    n_splits=5,
    device=None
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_mae_list = []

    train_data = np.array(train_data, dtype=object)

    for fold, (train_index, val_index) in enumerate(kf.split(train_data), start=1):
        print(f"\nFold {fold}/{n_splits}")

        train_split = train_data[train_index]
        val_split = train_data[val_index]

        train_user_tensor = torch.tensor(
            [user_to_idx[row[0]] for row in train_split],
            dtype=torch.long
        )

        train_item_tensor = torch.tensor(
            [item_to_idx[row[1]] for row in train_split],
            dtype=torch.long
        )

        train_rating_tensor = torch.tensor(
            [row[2] for row in train_split],
            dtype=torch.float32
        )

        model = MatrixFactorization(num_users, num_items, num_factors)

        model = train_model_fixed_epochs(
            model,
            train_user_tensor,
            train_item_tensor,
            train_rating_tensor,
            num_epochs=num_epochs,
            lr=lr,
            device=device
        )

        model.eval()

        predictions = []
        true_ratings = []

        for user_id, item_id, rating in val_split:
            pred = predict_rating(
                model,
                user_id,
                item_id,
                user_to_idx,
                item_to_idx,
                user_avg_dict,
                item_avg_dict,
                global_avg,
                device
            )

            predictions.append(pred)
            true_ratings.append(rating)

        fold_mae = np.mean(
            np.abs(np.array(predictions) - np.array(true_ratings))
        )

        fold_mae_list.append(fold_mae)

        print(f"Fold {fold} MAE: {fold_mae:.6f}")

    average_mae = np.mean(fold_mae_list)

    print("\nCross-validation results:")
    print("Fold MAEs:", fold_mae_list)
    print("Average MAE:", average_mae)

    return average_mae, fold_mae_list
    


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
average_mae, fold_mae_list = cross_validate_mf(
    train_data=train_data,
    num_users=num_users,
    num_items=num_items,
    user_to_idx=user_to_idx,
    item_to_idx=item_to_idx,
    user_avg_dict=user_avg_dict,
    item_avg_dict=item_avg_dict,
    global_avg=global_avg,
    num_factors=25,
    num_epochs=28056,
    lr=0.01,
    n_splits=5,
    device=device
)


Fold 1/5
Epoch 1000/28056, Train Loss: 0.942566
Epoch 2000/28056, Train Loss: 0.906606
Epoch 3000/28056, Train Loss: 0.884613
Epoch 4000/28056, Train Loss: 0.868726
Epoch 5000/28056, Train Loss: 0.856295
Epoch 6000/28056, Train Loss: 0.846097
Epoch 7000/28056, Train Loss: 0.837465
Epoch 8000/28056, Train Loss: 0.829993
Epoch 9000/28056, Train Loss: 0.823415
Epoch 10000/28056, Train Loss: 0.817546
Epoch 11000/28056, Train Loss: 0.812254
Epoch 12000/28056, Train Loss: 0.807440
Epoch 13000/28056, Train Loss: 0.803029
Epoch 14000/28056, Train Loss: 0.798960
Epoch 15000/28056, Train Loss: 0.795188
Epoch 16000/28056, Train Loss: 0.791674
Epoch 17000/28056, Train Loss: 0.788388
Epoch 18000/28056, Train Loss: 0.785303
Epoch 19000/28056, Train Loss: 0.782398
Epoch 20000/28056, Train Loss: 0.779655
Epoch 21000/28056, Train Loss: 0.777059
Epoch 22000/28056, Train Loss: 0.774595
Epoch 23000/28056, Train Loss: 0.772252
Epoch 24000/28056, Train Loss: 0.770020
Epoch 25000/28056, Train Loss: 0.767890

In [14]:
def predict_rating(
    model, 
    user_id, 
    item_id, 
    user_to_idx, 
    item_to_idx,
    user_avg_dict, 
    item_avg_dict,
    global_avg,
    device=None
):

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.eval()

    if user_id not in user_to_idx and item_id not in item_to_idx:
        return global_avg

    if user_id not in user_to_idx:
        return item_avg_dict.get(item_id, 0.5)

    if item_id not in item_to_idx:
        return user_avg_dict.get(user_id, 0.5)

    user_id_tensor = torch.tensor([user_to_idx[user_id]], dtype=torch.long).to(device)
    item_id_tensor = torch.tensor([item_to_idx[item_id]], dtype=torch.long).to(device)

    with torch.no_grad():
        prediction = model(user_id_tensor, item_id_tensor).item()

    prediction = round(prediction * 2) / 2
    prediction = max(0.5, min(prediction, 5.0))

    return prediction

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MatrixFactorization(
    num_users=num_users,
    num_items=num_items,
    num_factors=25
)

trained_model = train_model_fixed_epochs(
    model=model,
    train_user_tensor=train_user_tensor,
    train_item_tensor=train_item_tensor,
    train_rating_tensor=train_rating_tensor,
    num_epochs=28006,
    lr=0.01,
    device=device
)

torch.save({
    "model_state_dict": trained_model.state_dict(),
    "num_users": num_users,
    "num_items": num_items,
    "num_factors": 25
}, "best_mf_model_full_traindata.pth")

print("Model saved successfully.")

Epoch 1000/28006, Train Loss: 0.942043
Epoch 2000/28006, Train Loss: 0.906220
Epoch 3000/28006, Train Loss: 0.884398
Epoch 4000/28006, Train Loss: 0.868631
Epoch 5000/28006, Train Loss: 0.856285
Epoch 6000/28006, Train Loss: 0.846152
Epoch 7000/28006, Train Loss: 0.837573
Epoch 8000/28006, Train Loss: 0.830144
Epoch 9000/28006, Train Loss: 0.823602
Epoch 10000/28006, Train Loss: 0.817765
Epoch 11000/28006, Train Loss: 0.812501
Epoch 12000/28006, Train Loss: 0.807712
Epoch 13000/28006, Train Loss: 0.803322
Epoch 14000/28006, Train Loss: 0.799272
Epoch 15000/28006, Train Loss: 0.795517
Epoch 16000/28006, Train Loss: 0.792019
Epoch 17000/28006, Train Loss: 0.788748
Epoch 18000/28006, Train Loss: 0.785676
Epoch 19000/28006, Train Loss: 0.782784
Epoch 20000/28006, Train Loss: 0.780052
Epoch 21000/28006, Train Loss: 0.777467
Epoch 22000/28006, Train Loss: 0.775014
Epoch 23000/28006, Train Loss: 0.772681
Epoch 24000/28006, Train Loss: 0.770459
Epoch 25000/28006, Train Loss: 0.768339
Epoch 260

In [17]:
def test_data_predict_ratings(
    model, 
    test_data, 
    user_to_idx, 
    item_to_idx,
    user_avg_dict, 
    item_avg_dict, 
    global_avg, 
    device
):
    predictions = []
    for user_id, item_id, _ in test_data:
        rating = predict_rating(
            model, user_id, item_id,
            user_to_idx, item_to_idx,
            user_avg_dict, item_avg_dict,
            global_avg,
            device
        )
        predictions.append(rating)
    return predictions

def save_predictions_to_csv(predictions, test_data, output_file):
    with open(output_file, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        for (user_id, item_id, timestamp), rating in zip(test_data, predictions):
            writer.writerow([user_id, item_id, rating, timestamp])

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loaded_model_data = torch.load(
    "best_mf_model_full_traindata.pth",
    map_location=device
)

loaded_model = MatrixFactorization(
    loaded_model_data['num_users'],
    loaded_model_data['num_items'],
    loaded_model_data['num_factors']
)

loaded_model.load_state_dict(
    loaded_model_data['model_state_dict']
)

loaded_model = loaded_model.to(device)

loaded_model.eval()

predictions = test_data_predict_ratings(
    loaded_model,
    test_data,
    user_to_idx,
    item_to_idx,
    user_avg_dict,
    item_avg_dict,
    global_avg,
    device
)

output_file = 'Test20M_predicted_results.csv'

save_predictions_to_csv(
    predictions,
    test_data,
    output_file
)